In [ ]:
import pandas as pd 

# Load combined cleaned pathology data frame
df_path = r"D:\DATA\full_dataset\df_cleaned.csv"
df_selected = pd.read_csv(df_path)
print(df_selected.columns)

In [ ]:
from helper_functions import strings2lists

list_str_cols = ["snomed_code", "M", "T", "snomed_text", "T_text", "M_text", "undersoeger_anonymous"]

for col in list_str_cols: 
    df_selected[col] = df_selected[col].apply(strings2lists)

In [ ]:
from snomed_hierarchy import SNOMEDCodes, SNOMEDHierarchy

# Load SNOMED codes
snomed_path = "D:\DATA\patoSnoMed_2025-04.xlsx"
df_snomed = pd.read_excel(snomed_path)
snomed = SNOMEDCodes(df_snomed)

In [ ]:
# Create morphology hierarchy
broad_M = False

# Get df of M codes
m_codes = snomed.get_codes_by_letter('M')
all_m_codes = df_selected["M"].explode().unique()
m_filtered = m_codes[m_codes["SKSkode"].isin(all_m_codes)]

# Build hierarchy
if broad_M: 
    M_len = 2
else: 
    M_len = 3
m_hierarchy = SNOMEDHierarchy(m_filtered, main_len=M_len)

print("Number of available m codes: ", len(m_codes))
print("Number of m codes present in data set: ", len(m_filtered))

In [ ]:
# Create topography hierarchy
broad_T = True

# Get df of T codes
t_codes = snomed.get_codes_by_letter('T')
all_t_codes = df_selected["T"].explode().unique()
t_filtered = t_codes[t_codes["SKSkode"].isin(all_t_codes)]

# Build hierarchy
t_hierarchy = SNOMEDHierarchy(t_filtered, main_len=3)

print("Number of available t codes: ", len(t_codes))
print("Number of t codes present in data set: ", len(t_filtered))

In [ ]:
# t_hierarchy.print_all_regions(edited=False)

In [ ]:
# Manually edit T hierarchy
if broad_T: 
    # Update region name
    t_hierarchy.update_region('T00', 'Unspecific Topography')

    # Split T1X into T1X0 (Bløddelsvæv) and T1X+ (Knogle- og bruskvæv)
    t_hierarchy.split_main_region('T1X', {'T1X0': ['T1X0'], 'T1X+': ['T1X5','T1X7']})

    # Merge regions T01, T02 and T03 into one main region T01+
    t_hierarchy.merge_main_regions('T01+', ['T01', 'T02', 'T03', 'T1X0'], new_name="Skin, Subcutis and Appendages")

    # Other updates
    t_hierarchy.update_region('T04', 'Breast')
    t_hierarchy.merge_main_regions('T06+', ['T06','T07','T08', 'T09','T40', 'T45', 'T46', 'T48'], new_name="Blood, Bone Marrow and Lymphatic System")
    t_hierarchy.merge_main_regions('T10+', ['T10', 'T11', 'T1X+','T12','T13','T16','T17','T18'], new_name="Skeletal System")
    t_hierarchy.merge_main_regions('T21+', ['T21', 'T22','T23','T24'], new_name="Upper Respiratory Tract (ENT)")
    t_hierarchy.merge_main_regions('T25+', ['T25', 'T26','T28','T29','T2Y'], new_name="Lower Respiratory Tract")
    t_hierarchy.update_region('T34', 'Heart')
    t_hierarchy.merge_main_regions('T51+', ['T51','T52','T53','T54','T55','T60','T61','T62','T63'], new_name="Upper Digestive Tract")
    t_hierarchy.merge_main_regions('T64+', ['T64','T65','T66','T67','T68','T69'], new_name="Lower Digestive Tract")
    t_hierarchy.merge_main_regions('T56+', ['T56','T57','T59'], new_name="Accessory Digestive Organs")
    t_hierarchy.merge_main_regions('T71+', ['T71', 'T72','T73','T74','T75','T7X'], new_name="Urinary System")
    t_hierarchy.merge_main_regions('T76+', ['T76', 'T77','T78','T79'], new_name="Male Reproductive System")
    t_hierarchy.merge_main_regions('T80+', ['T80', 'T81','T82','T83','T84','T85','T86','T87','T8X'], new_name="Female Reproductive System (Maternal)")
    t_hierarchy.merge_main_regions('T88+', ['T88', 'T89'], new_name="Placenta, Fetal Membranes, and Fetus")
    t_hierarchy.merge_main_regions('T93+', ['T93', 'T96','T97','T98'], new_name="Endocrine System")
    t_hierarchy.merge_main_regions('TX2+', ['TX2', 'TX9'], new_name="Nervous System")
    t_hierarchy.merge_main_regions('TY0+', ['TY0', 'TXX','TXY'], new_name="Region: Head and Neck")
    t_hierarchy.merge_main_regions('TY1+', ['TY1', 'TY2','TY4','TY6','TY7'], new_name="Region: Trunk")
    t_hierarchy.update_region('TY8', 'Region: Upper Extremity')
    t_hierarchy.update_region('TY9', 'Region: Lower Extremity')

else: 
    # Update region name
    t_hierarchy.update_region('T00', 'Uspecifik topografi')

    # Merge regions T01, T02 and T03 into one main region T01+
    t_hierarchy.merge_main_regions('T01+', ['T01', 'T02', 'T03'], new_name="Hud inkl. subcutis")

    # Split T1X into T1X0 (Bløddelsvæv) and T1X+ (Knogle- og bruskvæv)
    t_hierarchy.split_main_region('T1X', {'T1X0': ['T1X0'], 'T1X+': ['T1X5','T1X7']})

    # Other updates
    t_hierarchy.update_region('T04', 'Mamma')
    t_hierarchy.merge_main_regions('T08+', ['T08', 'T09'], new_name="Lymfeknude og lymfekar")
    t_hierarchy.merge_main_regions('T10+', ['T10', 'T11', 'T1X+'], new_name="Knogle")
    t_hierarchy.update_region('T24', 'Larynx')
    t_hierarchy.split_main_region('T2Y', {'T2Y4': ['T2Y4'], 'T2Y6': ['T2Y6']})
    t_hierarchy.merge_main_regions('T26+', ['T26', 'T2Y4'], new_name="Bronchus")
    t_hierarchy.merge_main_regions('T29+', ['T29', 'T2Y6'], new_name="Pleura")
    t_hierarchy.merge_main_regions('T40+', ['T40', 'T45', 'T46', 'T48'], new_name="Blodkar")
    t_hierarchy.update_region('T67', 'Colon')
    t_hierarchy.update_region('T68', 'Rectum')
    t_hierarchy.merge_main_regions('T71+', ['T71', 'T72'], new_name="Nyrer")
    t_hierarchy.merge_main_regions('T74+', ['T74', 'T7X'], new_name="Urinblære")
    t_hierarchy.update_region('T79', 'Øvrige hanlige kønsorganer')
    t_hierarchy.merge_main_regions('T83+', ['T83','T8X'], new_name="Cervix Uteri")
    t_hierarchy.merge_main_regions('T88+', ['T88','T89'])
    t_hierarchy.update_region('T93', 'Binyre')
    t_hierarchy.update_region('TX2', 'Hjerne')
    t_hierarchy.update_region('TXX', 'Øje')
    t_hierarchy.update_region('TY0', 'Region: hoved og hals')
    t_hierarchy.update_region('TY1', 'Region: truncus')
    t_hierarchy.update_region('TY2', 'Region: thorax')
    t_hierarchy.update_region('TY4', 'Region: abdomen')
    t_hierarchy.update_region('TY6', 'Region: pelvis')
    t_hierarchy.update_region('TY7', 'Region: inguen')
    t_hierarchy.update_region('TY8', 'Region: overekstremitet')
    t_hierarchy.update_region('TY9', 'Region: underekstremitet')

In [ ]:
t_hierarchy.list_main_regions(edited = True)

In [ ]:
t_hierarchy.print_all_regions(edited=True)

In [ ]:
m_hierarchy.print_all_regions(edited=False)

In [ ]:
# Manually edit M hierarchy

if broad_M: 
    m_hierarchy.update_region('M0', 'Unspecific morphology')
    m_hierarchy.update_region('M1', 'Traumatic changes')
    m_hierarchy.update_region('M2', 'Congenital malformations, pregnancy products')
    m_hierarchy.update_region('M3', 'Mechanical changes')
    m_hierarchy.update_region('M4', 'Inflammation and fibrosis')
    m_hierarchy.update_region('M5', 'Degeneration, necrosis, deposition, dystrophy, atrophy') 
    m_hierarchy.update_region('M6', 'Cellular changes') 
    m_hierarchy.update_region('M7', 'Growth and maturation changes') 
    m_hierarchy.merge_main_regions('M8-9', ['M8', 'M9'], new_name="Neoplasms")
    m_hierarchy.update_region('MÆ', 'Description in text')

else:                                                                                                       
# Unspecific morphology
    m_hierarchy.split_main_region('M00', {'M000': ['M000'], 'M00+': ['M001','M004']})
    m_hierarchy.split_main_region('M09', {'M090+': ['M090', 'M091'], 'M094': ['M094']})
    m_hierarchy.merge_main_regions('M000+', ['M000', 'MÆ0', 'M090+'], new_name='Morphology Not Applicable / Insufficient Tissue')  

# Resection Margin
    m_hierarchy.fine_split('M094', 
                           {'M09400+': ['M09400', 'M09405', 'M09413', 'M09416','M09420', 'M09450', 'M09451', 'M09453', 'M09462', 'M09463', 'M09470'],  # Resection Margin Free
                           'M09401+': ['M09401','M09406', 'M09414', 'M09417', 'M09421', 'M09431'],  # Not Free
                           'M09402+': ['M09402','M09415','M09418']}) # Uncertain 
    m_hierarchy.update_region('M09400+', 'Resection Margin Free')
    m_hierarchy.update_region('M09401+', 'Resection Margin Not Free')
    m_hierarchy.update_region('M09402+', 'Resection Margin Uncertain')
     
# Traumatic changes
    m_hierarchy.merge_main_regions('M10+', ['M10', 'M11', 'M12', 'M14', 'M18'], new_name='Traumatic Lesions')

# Congenital malformations, pregnancy products
    m_hierarchy.merge_main_regions('M20+', ['M20', 'M21','M22', 'M26', 'M75'], new_name="Congenital Malformations")
    m_hierarchy.merge_main_regions('M28+', ['M28', 'M29', 'M79'], new_name='Pregnancy-Related Tissues/Changes')

# Mechanical changes
    m_hierarchy.merge_main_regions('M30+', ['M30', 'M33', 'M35', 'M36', 'M37', 'M52'], new_name='Obstruction / Fluid Retention / Cysts')
    m_hierarchy.merge_main_regions('M38+', ['M31', 'M32', 'M34', 'M38', 'M39'], new_name='Mechanical Changes / Architectural Distortion')
    
# Inflammation and fibrosis
    m_hierarchy.merge_main_regions('M40+', ['M40', 'M41', 'M42', 'M43', 'M44', 'M45', 'M46', 'M47','M48'], new_name='Inflammation')
    m_hierarchy.update_region('M49', 'Fibrosis')
    
# Degeneration, necrosis, deposition, dystrophy, atrophy
    m_hierarchy.split_main_region('M70', {'M703': ['M703'], 'M708': ['M708']})
    m_hierarchy.merge_main_regions('M50+', ['M50', 'M51', 'M54', 'M65', 'M58', 'M708'], new_name='Degeneration / Necrosis / Atrophy')
    m_hierarchy.merge_main_regions('M55+', ['M55', 'M57'], new_name="Material Deposits")
    
# Cellular changes
    m_hierarchy.split_main_region('M69', {'M692+': ['M692', 'M697'], 'M698': ['M698']})        
    m_hierarchy.merge_main_regions('M00++', ['M00+', 'M698', 'M703'], new_name="Normal Tissue")
    m_hierarchy.merge_main_regions('M01+', ['M01', 'M02', 'M63', 'M66', 'M692+'], new_name="Cellular Changes / Abnormal Tissue Structure")

# Growth and maturation changes
    m_hierarchy.merge_main_regions('M71+', ['M71', 'M72', 'M73', 'M74', 'M76', 'M77'], new_name='Proliferative/Pre-neoplastic Changes')
        
# Neoplasms
    m_hierarchy.merge_main_regions('M8+', ['M80','M81', 'M81', 'M82', 'M83', 'M84', 'M85', 'M86','M87', 'M88', 'M89', 'M90','M91', 'M92', 'M95', 'M96', 'M97','M98', 'M99'], new_name='Neoplasms')    
    m_hierarchy.split_main_region_by_suffix('M8+', {'M_0': 'Benign Neoplasm', 'M_1': 'Uncertain / Borderline Neoplasm', 'M_2': 'In Situ Neoplasm', 'M_3': 'Malignant 3', 'M_6': 'Malignant 6', 'M_7': 'Malignant 7', 'M_9': 'Malignant 9'})
    m_hierarchy.merge_main_regions('M_3+', ['M_3', 'M_6', 'M_7','M_9','M8+'], new_name='Malignant Neoplasm')


In [ ]:
m_hierarchy.list_main_regions(edited = True)

In [ ]:
m_hierarchy.print_all_regions(edited=True)

In [ ]:
def map_codes_to_category(codes, hierarchy):
    categories = []
    for code in codes:
        region_name = hierarchy.code_to_main_region_name(code)
        if region_name is None:
            print("Original code:", code)
            print("In t_codes:", code in t_codes["SKSkode"].values)
            print("In t_filtered:", code in t_codes["SKSkode"].values)
            print("In m_codes:", code in m_codes["SKSkode"].values)
            print("In m_filtered:", code in t_codes["SKSkode"].values)
        categories.append(region_name)
    return list(set(categories))

In [ ]:
df_selected["T_category"] = df_selected["T"].apply(lambda x: map_codes_to_category(x, t_hierarchy))
df_selected["M_category"] = df_selected["M"].apply(lambda x: map_codes_to_category(x, m_hierarchy))

In [ ]:
snomed_dict = snomed.code_to_text()

def map_codes_to_text(codes, snomed_dict):
    texts = []
    for code in codes:
        text = snomed_dict.get(code)
        if text is None:
            print("Original code:", code)
        texts.append(text)
    return list(set(texts))

In [ ]:
print(df_selected.head())

In [ ]:
# Save to csv
output_file = r"D:\DATA\full_dataset\with_snomed_category.csv"

df_selected.to_csv(output_file, index=False)
print(f"Saved DataFrame to {output_file}")

In [ ]:
# SNOMED counts for figure
cols_to_keep = ['snomed_code']
df_fig1 = pd.DataFrame(df_selected, columns=cols_to_keep)
print(df_fig1.head())

In [ ]:
def get_all(df, col):
    no_na = df[col].dropna()
    all_codes = [code for c in no_na for code in c]
    return all_codes

all_codes = get_all(df_fig1, "snomed_code")
print(all_codes[:10])

In [ ]:
codes_df = pd.DataFrame({"code": all_codes})

In [ ]:
codes_df["prefix"] = codes_df["code"].astype(str).str[0]

In [ ]:
prefix_names = {
    'M': 'Morphology',
    'T': 'Topography',
    'F': 'Function',
    'P': 'Procedure',
    'Æ': 'Etiology',
    'S': 'Disease'
}

In [ ]:
def code_to_category(code, prefix_map):
    prefix = code[0]
    if prefix in ["T"]:
        category = t_hierarchy.code_to_main_region_name(code)
    elif prefix in ["M"]:
        category = m_hierarchy.code_to_main_region_name(code)
    else: 
        category = prefix_map[prefix]
    return category

In [ ]:
codes_df["category"] = codes_df["code"].apply(
    lambda x: code_to_category(x, prefix_names)
)

In [ ]:
print(codes_df.head())

In [ ]:
result = (
    codes_df
    .groupby(["prefix", "category"])
    .size()
    .reset_index(name="count")
)
print(result.head())

In [ ]:
# Save to csv
output_file = r"D:\DATA\full_dataset\snomed_counts.csv"

result.to_csv(output_file, index=False)
print(f"Saved DataFrame to {output_file}")